# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mynhi1011/Flyrank-ML.Project/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.



### 1. My lane as an ML task

- **Chosen Lane:** Lane 2 — Refresh / Content Opportunity Scoring
- **Task Type:** Ranking / Scoring (Priority Scoring)

**Framing:**
The task is framed as a ranking/scoring problem where the model outputs an opportunity score (or probability of needing review/refresh) for each page. Rather than a purely binary accept/reject decision, the output produces a prioritized queue so that content editors and SEO specialists can review the highest-priority pages first within their limited weekly review bandwidth.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### 2. Target or Proxy

- **Target / Proxy Concept:** A binary or continuous indicator reflecting that a page has experienced significant performance decline or underperformed relative to its historical baseline and search volume potential.
- **Proxy Definition in Starter Data:** In this observational dataset, we define a proxy target `needs_review` (or `is_declining`) representing pages whose recent 90-day impressions are substantially below their potential, or pages flagged with a negative trend that warrant content refresh intervention.
- **Why it is a proxy:** We do not observe ground-truth editor review decisions in the raw log data; therefore, observable drops in engagement/impressions serve as an honest proxy for refresh need.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### 3. Success Metric

- **Primary Metric:** **Precision@K** (specifically **Precision@50** or **Precision@100**)
- **Why this metric matters:**
  Editorial teams have finite human bandwidth (e.g., they can only manually inspect 50–100 pages per week). A global metric like standard ROC-AUC or overall accuracy gives equal weight to all 30,000 pages, most of which will never be opened. What matters in production is that when an editor reviews the top $K$ recommended pages, as many of them as possible truly need refresh actions (minimizing wasted reviewer hours).

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### 4. Unit of Analysis

- **Unit of Analysis:** **One URL / Page** (1 row = 1 unique content page).
- **Features available per page:** Historical impressions (`impressions_90d`), click-through rate (`ctr`), search volume (`search_volume`), and content metadata signals.
- **Target sketch:** A proxy flag `needs_review` constructed to demonstrate what the supervised or scoring target column looks like on actual data.

In [5]:
import os
import pandas as pd
import numpy as np

if not os.path.exists("Flyrank-ML.Project"):
    !git clone https://github.com/mynhi1011/Flyrank-ML.Project.git
else:
    print("Repo already available.")

df = pd.read_csv("Flyrank-ML.Project/data/raw/content_refresh_anonymized.csv")

high_volume_threshold = df["search_volume"].quantile(0.50)
low_imp_threshold = df["impressions_90d"].quantile(0.25)

df["target_needs_review"] = (
    (df["search_volume"] >= high_volume_threshold) &
    (df["impressions_90d"] <= low_imp_threshold)
).astype(int)

print(f"Total rows (pages): {len(df)}")
print(f"Target distribution (needs_review):\n{df['target_needs_review'].value_counts(normalize=True).round(3)}")

display_cols = [c for c in ["url", "search_volume", "impressions_90d", "ctr", "target_needs_review"] if c in df.columns]
df[display_cols].head()

Repo already available.
Total rows (pages): 30000
Target distribution (needs_review):
target_needs_review
0    0.89
1    0.11
Name: proportion, dtype: float64


,search_volume,impressions_90d,ctr,target_needs_review
0,10.0,3803,0.76,0
1,90.0,15320,0.05,0
2,0.0,12581,0.09,0
3,10.0,11751,0.49,0
4,0.0,19140,0.13,0


### 5. Why ML beats a fixed rule here

- **Limitation of simple rules:** A fixed rule (e.g., `impressions < 500` or `ctr < 0.02`) fails because pages operate across vastly different search volume tiers and content topics. A single static threshold creates massive false-positive lists or misses declining pages in high-volume categories.
- **Advantage of ML/Priority Scoring:** Machine learning can combine multi-dimensional signals simultaneously—search demand, content age/length signals, and performance interactions. Instead of a brittle binary cutoff, ML computes calibrated priority scores that dynamically surface the strongest candidates for human review.
- **Real Content Action:** The output feeds an actionable weekly queue where content teams execute targeted actions: refresh outdated content, expand high-potential sections, or monitor stable assets.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 6. Self-check

- [x] I named the ML task type (Ranking / Scoring).
- [x] I named the target / proxy and explained why it is a proxy.
- [x] I identified a production-relevant success metric (Precision@K).
- [x] I showed the unit of analysis as a real dataframe (1 row = 1 page).
- [x] I sketched what the target column looks like on real data.
- [x] I explained why ML beats a fixed heuristic rule.
- [x] I tied the prediction output to an editorial action (refresh queue).